In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from skorch import NeuralNetClassifier
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as torchdata
from astroML.utils import completeness_contamination

We want to setup a neural network to classify particle collision events into sources (Higgs boson productions) and background 

In [2]:
#loading the data
data = pd.read_csv('../../solutions/higgs/higgs.csv')
data

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,349995,-999.000,71.989,36.548,5.042,-999.00,-999.000,-999.000,1.392,5.042,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,4.505083,b
249996,349996,-999.000,58.179,68.083,22.439,-999.00,-999.000,-999.000,2.585,22.439,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,2.497259,b
249997,349997,105.457,60.526,75.839,39.757,-999.00,-999.000,-999.000,2.390,22.183,...,1,41.992,1.800,-0.166,-999.000,-999.00,-999.000,41.992,0.018636,s
249998,349998,94.951,19.362,68.812,13.504,-999.00,-999.000,-999.000,3.365,13.504,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,1.681611,b


In [3]:
data.keys()

Index(['EventId', 'DER_mass_MMC', 'DER_mass_transverse_met_lep',
       'DER_mass_vis', 'DER_pt_h', 'DER_deltaeta_jet_jet', 'DER_mass_jet_jet',
       'DER_prodeta_jet_jet', 'DER_deltar_tau_lep', 'DER_pt_tot', 'DER_sum_pt',
       'DER_pt_ratio_lep_tau', 'DER_met_phi_centrality',
       'DER_lep_eta_centrality', 'PRI_tau_pt', 'PRI_tau_eta', 'PRI_tau_phi',
       'PRI_lep_pt', 'PRI_lep_eta', 'PRI_lep_phi', 'PRI_met', 'PRI_met_phi',
       'PRI_met_sumet', 'PRI_jet_num', 'PRI_jet_leading_pt',
       'PRI_jet_leading_eta', 'PRI_jet_leading_phi', 'PRI_jet_subleading_pt',
       'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi', 'PRI_jet_all_pt',
       'Weight', 'Label'],
      dtype='object')

For simplicity we consider only the events without missing features

In [4]:
idx_list = [] # Id of the events without missing feats
for i in range(len(data)):
    if (data['DER_deltaeta_jet_jet'][i] != -999.00 
        and data['PRI_jet_leading_pt'][i] != -999.00 
        and data['PRI_jet_subleading_pt'][i] != -999.00):
        idx_list.append(i)

#cleaning the data
data_cleaned = [data.iloc[i] for i in idx_list]
data_cleaned = np.array(data_cleaned)
datac_len = len(data_cleaned[:, 0])
data_cleaned.shape

(72543, 33)

In [5]:
X_scaled = np.zeros((datac_len, 30), dtype=np.float32)

# scaling the data to have zero mean and unit variance in each feature
for i in range(1, 31):
    X_scaled[:, i-1] = (data_cleaned[:, i] - data_cleaned[:, i].mean())/data_cleaned[:, i].std()

y = [int(data_cleaned[i, -1]=='s') for i in range(datac_len)] #labels [1=source, 0=background]
y = np.array(y, dtype=np.float32)


## MLP

We set up a MLP with three hidden layers, RELU activation functions and dropout regularization

In [6]:
class MLP(nn.Module):
    def __init__(self, nhidden1, nhidden2, nhidden3, dp_rate):
        super(MLP, self).__init__()
        self.fc_1 = nn.Linear(30, nhidden1)
        self.drop1 = nn.Dropout(dp_rate)
        self.fc_2 = nn.Linear(nhidden1, nhidden2)
        self.drop2 = nn.Dropout(dp_rate)
        self.fc_3 = nn.Linear(nhidden2, nhidden3)
        self.drop3 = nn.Dropout(dp_rate)
        self.fc_o = nn.Linear(nhidden3, 1)

    def forward(self, x):
        x = F.relu(self.fc_1(x))
        x = self.drop1(x)
        x = F.relu(self.fc_2(x))
        x = self.drop2(x)
        x = F.relu(self.fc_3(x))
        x = self.drop3(x)
        x = self.fc_o(x)
        return x

We split the data into training and test sets, we use a subset of the training split to tune the hyperparameters of the network via k-fold cross validation (we don't use the full training set to decrease the computational time)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, train_size=0.8)

In [8]:
#taking a fifth of the training set for CV
X_CV = X_train[::5]
y_CV = y_train[::5][:, None]

In [ ]:
sk_net = NeuralNetClassifier(
    MLP,
    max_epochs=30,
    optimizer=torch.optim.Adam,
    optimizer__weight_decay = 1e-4,
    criterion=nn.BCEWithLogitsLoss,
    train_split=None
)

# hyperparameter grid considered during CV
param_dist = {
    'lr': [5e-4, 1e-3, 5e-3],
    'module__nhidden1': [64, 128, 256],
    'module__nhidden2': [64, 128, 256],
    'module__nhidden3': [32, 64, 128],
    'module__dp_rate': [0.1, 0.15, 0.2],
    'batch_size': [128, 256]
}
k = 5 # number of folds

# cross validation
CV = RandomizedSearchCV(sk_net, param_distributions=param_dist,
                        scoring='accuracy', cv=k, n_iter=50)
CV.fit(X_train, y_train[:, None])



  epoch    train_loss     dur
-------  ------------  ------
      1        0.5392  2.3773
      2        0.4246  2.6357
      3        0.3965  2.2233
      4        0.3828  2.1775
      5        0.3768  2.2989
      6        0.3727  2.4406
      7        0.3669  2.1028
      8        0.3646  2.2992
      9        0.3613  2.0811
     10        0.3578  2.2649
     11        0.3556  2.1930
     12        0.3533  2.5219
     13        0.3508  2.5179
     14        0.3491  2.4524
     15        0.3466  2.4243
     16        0.3449  2.2887
     17        0.3432  2.3827
     18        0.3420  2.2870
     19        0.3385  2.3515
     20        0.3397  2.2449
     21        0.3377  2.1119
     22        0.3355  2.4971
     23        0.3355  2.1174
     24        0.3344  2.3400
     25        0.3322  2.3067
     26        0.3306  2.3056
     27        0.3316  2.3224
     28        0.3306  2.2185
     29        0.3292  2.3293
     30        0.3278  2.2024
  epoch    train_loss     dur
-------  -

,estimator,"<class 'skorc...ain__.MLP'>, )"
,param_distributions,"{'batch_size': [128, 256], 'lr': [0.0005, 0.001, ...], 'module__dp_rate': [0.1, 0.15, ...], 'module__nhidden1': [64, 128, ...], ...}"
,n_iter,50
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [15]:
param_grid = {
    'lr': [CV.best_params_['lr']],
    'module__nhidden1': [64, 128, 256],
    'module__nhidden2': [64, 128, 256],
    'module__nhidden3': [32, 64, 128],
    'module__dp_rate': [CV.best_params_['module__dp_rate']],
    'batch_size': [CV.best_params_['batch_size']]
}
k_grid = 3 
CV_grid = GridSearchCV(sk_net, param_grid=param_grid, scoring='accuracy', cv=k_grid)

CV_grid.fit(X_train, y_train[:, None])

  epoch    train_loss     dur
-------  ------------  ------
      1        0.5568  2.3099
      2        0.4388  2.4004
      3        0.4104  2.0939
      4        0.3996  1.9398
      5        0.3919  2.1755
      6        0.3867  1.7839
      7        0.3817  1.8453
      8        0.3798  1.9097
      9        0.3758  1.4717
     10        0.3736  1.9291
     11        0.3711  1.8696
     12        0.3695  1.6778
     13        0.3661  1.9937
     14        0.3662  1.8553
     15        0.3654  1.9326
     16        0.3631  1.8275
     17        0.3618  1.9949
     18        0.3598  2.0340
     19        0.3571  1.9450
     20        0.3586  1.7411
     21        0.3568  1.9387
     22        0.3580  1.7175
     23        0.3540  1.7939
     24        0.3557  1.7708
     25        0.3555  1.7258
     26        0.3548  1.5308
     27        0.3531  1.8477
     28        0.3519  1.7805
     29        0.3500  2.0223
     30        0.3508  1.8708
  epoch    train_loss     dur
-------  -

,estimator,"<class 'skorc...ain__.MLP'>, )"
,param_grid,"{'batch_size': [256], 'lr': [0.001], 'module__dp_rate': [0.2], 'module__nhidden1': [64, 128, ...], ...}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,module,<class '__main__.MLP'>


In [16]:
CV_grid.best_params_

{'batch_size': 256,
 'lr': 0.001,
 'module__dp_rate': 0.2,
 'module__nhidden1': 64,
 'module__nhidden2': 64,
 'module__nhidden3': 32}

In [17]:
nh1 = CV_grid.best_params_['module__nhidden1']
nh2 = CV_grid.best_params_['module__nhidden2']
nh3 = CV_grid.best_params_['module__nhidden3']
lr = CV_grid.best_params_['lr']
dr = CV_grid.best_params_['module__dp_rate']
batch_size = CV_grid.best_params_['batch_size']

In [18]:
# converting the data to pytorch tensors

X_train_torch = torch.tensor(X_train)
y_train_torch = torch.tensor(y_train, dtype=torch.float32)

X_test_torch = torch.tensor(X_test)
y_test_torch = torch.tensor(y_test, dtype=torch.float32)

train_dataset = torchdata.TensorDataset(X_train_torch, y_train_torch.view(-1, 1))
test_dataset = torchdata.TensorDataset(X_test_torch, y_test_torch.view(-1, 1))

train_dataloader = torchdata.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

We now train the network with the optimized hyperparameters on the entire test set, evaluating performance on the test set every 10 epochs

In [ ]:
model = MLP(nh1, nh2, nh3, dr) # network

criterion = torch.nn.BCEWithLogitsLoss() # loss function (BCE with logits)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4) # parameter optimization (Adam with L2 loss)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,
                                                       verbose=True,
                                                       patience=3,
                                                       threshold=1e-3) # scheduler to adaptively optimize for the learning rate

epochs = 100 # number of epochs
no_impr_epochs = 0
min_valid_loss = float('inf')
patience = 5

#training
for t in range(epochs):

    train_loss = 0.
    for X_batch, y_batch in train_dataloader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.view(-1, 1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    with torch.no_grad():
        pred_logits = model(X_test_torch)
        valid_loss = criterion(pred_logits, y_test_torch.view(-1, 1))
        y_pred = (torch.sigmoid(pred_logits) > 0.5).float() # applying sigmoid activation fucntion on the output of the final layer

        if valid_loss > min_valid_loss*(1-1e-3):
                no_impr_epochs += 1
        else:
            min_valid_loss = valid_loss
            no_impr_epochs = 0
        if (no_impr_epochs == patience or t == epochs-1): # stopping training after 5 epochs without improvement
            accuracy = (y_pred.eq(y_test_torch.view(-1,1)).sum().item()) / len(y_test_torch)
            comp, cont = completeness_contamination(y_pred.numpy().ravel(), y_test_torch.numpy().ravel())
            print('Epoch: '+str(t+1))
            print('Training loss: '+f"{train_loss:.3f}"+' Test loss: '+f"{valid_loss.item():.3f}")
            print('Accuracy: '+f"{accuracy:.3f}"+' Completeness: '+f"{comp:.3f}"+' Contamination: '+f"{cont:.3f}")
            print('FINISHED TRAINING')
            break
    
    if ((t+1)%10 == 0): # evaluating performance every 10 epochs
        accuracy = (y_pred.eq(y_test_torch.view(-1,1)).sum().item()) / len(y_test_torch)
        comp, cont = completeness_contamination(y_pred.numpy().ravel(), y_test_torch.numpy().ravel())
        print('Epoch: '+str(t+1))
        print('Training loss: '+f"{train_loss:.3f}"+' Test loss: '+f"{valid_loss.item():.3f}")
        print('Accuracy: '+f"{accuracy:.3f}"+' Completeness: '+f"{comp:.3f}"+' Contamination: '+f"{cont:.3f}")
        print('---------------------------------------------------------------')
    scheduler.step(valid_loss.item())



/home/gabriele/anaconda3/envs/jupyter_env/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch: 10
Training loss: 82.638 Test loss: 0.359
Accuracy: 0.847 Completeness: 0.798 Contamination: 0.148
---------------------------------------------------------------
Epoch: 20
Training loss: 79.937 Test loss: 0.353
Accuracy: 0.847 Completeness: 0.829 Contamination: 0.171
---------------------------------------------------------------
Epoch: 26
Training loss: 79.206 Test loss: 0.353
Accuracy: 0.848 Completeness: 0.823 Contamination: 0.164
FINISHED TRAINING


## Random Forest

We compare the results to the ones obtained by using a random forest classifier

In [15]:
forest = RandomForestClassifier(n_estimators=300,
                                max_depth=None,
                                random_state=124)
forest.fit(train_dataset[:][0].numpy(), train_dataset[:][1].numpy())


/home/gabriele/anaconda3/envs/jupyter_env/lib/python3.11/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
y_pred_for = forest.predict(X_test)
comp_for, cont_for = completeness_contamination(y_pred_for, y_test)
print('Completeness: '+f"{comp_for:.3f}"+' Contamination: '+f"{cont_for:.3f}")

Completeness: 0.816 Contamination: 0.162
